## Different Methods and Basis Sets

In this session, we employ a number of electron correlation methods and investigate their impact on reaction energies. We also investigate the dependence on the basis set size.

### Exercises
1. Calculate reaction energy using RHF, MP2, CCD, CCSD, CCSD(T), CISD, PBE, TPSS, TPSSH and B3LYP with the cc-pVTZ basis set for the following reaction: 
$ \text{CO} + \text{H}_2 \rightarrow \text{H}_2\text{CO} $ 
2. Calculate the correction energies of a fluorine molecule $ \text{F}_2 $ using CCSD, CCSD(T), CCSD(T)-F12 with cc-pVDZ, cc-pVTZ, cc-pVQZ and cc-pV5Z basis set 

In [8]:
from ase.visualize import view
from ase.build import molecule
from ase.units import Hartree,kJ
from pymolpro import ASEMolpro
from ase.optimize import BFGS
#from ase.vibrations import Vibrations 
from ase import Atoms
import nglview as nv 
import matplotlib.pyplot as plt
from ase import units

### Exercise 1: $ \text{CO} + \text{H}_2 \rightarrow \text{H}_2\text{CO} $

The individual molecules involved in the reaction, $ \text{CO} $, $\text{H}_2 $ and $\text{H}_2\text{CO} $, are constructed using ASE. This provides the initial geometries for the subsequent calculations.

In [2]:
# build molecules using Atoms and molecule
CO = molecule("CO")
H2 = molecule("H2")
H2CO = molecule("H2CO")



In [ ]:
# view information about the molecules and visualize molecules (optional)


Next, the geometries of the molecules are optimized using MP2/cc-pVTZ. 

In [3]:
print("Geometry optimization of H2:")
print("")
H2.calc = ASEMolpro(ansatz='MP2/cc-pVTZ', density_fitting=True)
with BFGS(H2, trajectory='H2.traj') as opt:
    opt.run(fmax=0.0001)

Geometry optimization of H2:

      Step     Time          Energy          fmax
BFGS:    0 13:36:15      -31.691596        0.005736
BFGS:    1 13:36:20      -31.691596        0.000484
BFGS:    2 13:36:24      -31.691596        0.000000


In [4]:
print("Geometry optimization of CO:")
print("")
CO.calc = ASEMolpro(ansatz='MP2/cc-pVTZ', density_fitting=True)
with BFGS(CO, trajectory='CO.traj') as opt:
    opt.run(fmax=0.0001)

Geometry optimization of CO:

      Step     Time          Energy          fmax
BFGS:    0 13:36:32    -3078.565787        1.278266
BFGS:    1 13:36:37    -3078.536483        3.070593
BFGS:    2 13:36:42    -3078.573346        0.115041
BFGS:    3 13:36:47    -3078.573404        0.009841
BFGS:    4 13:36:52    -3078.573404        0.000036


In [5]:
print("Geometry optimization of H2CO:")
print("")
H2CO.calc = ASEMolpro(ansatz='MP2/cc-pVTZ', density_fitting=True)
with BFGS(H2CO, trajectory='H2CO.traj') as opt:
    opt.run(fmax=0.0001)

Geometry optimization of H2CO:

      Step     Time          Energy          fmax
BFGS:    0 13:37:03    -3110.448876        0.787715
BFGS:    1 13:37:09    -3110.448724        0.938654
BFGS:    2 13:37:16    -3110.453099        0.059788
BFGS:    3 13:37:23    -3110.453266        0.039153
BFGS:    4 13:37:29    -3110.453343        0.026036
BFGS:    5 13:37:36    -3110.453366        0.020214
BFGS:    6 13:37:42    -3110.453383        0.005333
BFGS:    7 13:37:49    -3110.453384        0.001057
BFGS:    8 13:37:55    -3110.453384        0.000046


After the geometry optimization, the energies of the molecules are computed using the methods RHF, MP2, CCD, CCSD, CCSD(T), CISD, PBE, TPSS, TPSSh, B3LYP with the basis set cc-pVTZ.

Note: a loop over the different methods can be used to automatize these calculations. 

Note 2: CCD must be treat as a special case of CCSD.  

ASEMolpro( 
                specification="""
                basis=cc-pVTZ
                {ccsd; option,ccd}
                """
            )

In [25]:
methods = ["RHF", "MP2", "CCD", "CCSD", "CCSD(T)", "CISD", "PBE", "TPSS", "TPSSh", "B3LYP"]
basis = 'cc-pVTZ'

results_h2 = {}
results_co = {}
results_h2co = {}

for i, method in enumerate(methods):
    print(f"Method {i+1}/{len(methods)}")
    
    if method == "CCD":
        H2.calc = ASEMolpro(specification="""basis=cc-pVTZ{ccsd; option,ccd}""")
        CO.calc = ASEMolpro(specification="""basis=cc-pVTZ{ccsd; option,ccd}""")
        H2CO.calc = ASEMolpro(specification="""basis=cc-pVTZ{ccsd; option,ccd}""")
    else: 
        H2.calc = ASEMolpro(ansatz=f"{method}/{basis}")
        CO.calc = ASEMolpro(ansatz=f"{method}/{basis}")
        H2CO.calc = ASEMolpro(ansatz=f"{method}/{basis}")

    # calculate potential energies and store in results dictionaries
    results_h2[method] = H2.get_potential_energy()
    results_co[method] = CO.get_potential_energy()
    results_h2co[method] = H2CO.get_potential_energy()

Method 1/10
Method 2/10
Method 3/10
Method 4/10
Method 5/10
Method 6/10
Method 7/10
Method 8/10
Method 9/10
Method 10/10


Finally, the reaction energy and the correlation energy can be determined.  

In [11]:
print("Potential energies in kJ/mol:")
for method in methods:
    print(method,"/",basis,": ", results_h2[method] * units.mol/units.kJ)

Potential energies in kJ/mol:
RHF / cc-pVTZ :  -2974.651508939508
MP2 / cc-pVTZ :  -3057.7875700702316
CCD / cc-pVTZ :  -2963.292550892191
CCSD / cc-pVTZ :  -3077.914233820179
CCSD(T) / cc-pVTZ :  -3077.914233820179
CISD / cc-pVTZ :  -3077.9142125739277
PBE / cc-pVTZ :  -3061.3964589565403
TPSS / cc-pVTZ :  -3097.5347384079237
TPSSh / cc-pVTZ :  -3096.2132782416047
B3LYP / cc-pVTZ :  -3080.281076548011


In [9]:
reaction_energy = {}

print("Reaction energies in kJ/mol:")
for method in methods:
    # calculate reaction energy in kJ/mol and store in reaction_energy dictionary
    reaction_energy[method] = (results_h2co[method] - results_h2[method] - results_co[method]) * units.mol/units.kJ

    print(method,"/",basis,": ", reaction_energy[method])

Reaction energies in kJ/mol:
RHF / cc-pVTZ :  1.6963879156200719
MP2 / cc-pVTZ :  -18.107287778185505
CCD / cc-pVTZ :  2.5059776514018064
CCSD / cc-pVTZ :  -17.233254545946718
CCSD(T) / cc-pVTZ :  -15.715192051543328
CISD / cc-pVTZ :  0.8805051135843426
PBE / cc-pVTZ :  -49.99721290364528
TPSS / cc-pVTZ :  -35.47060025614751
TPSSh / cc-pVTZ :  -37.61999274065803
B3LYP / cc-pVTZ :  -31.50996121683225


In [12]:
correlation_energy = {}

H2.calc = ASEMolpro(ansatz="HF/cc-pVTZ")
E_HF = H2.get_potential_energy()

print("Correlation energy of H2 in kJ/mol:")
for method in methods:
    # calculate correlation energy in kJ/mol and store in correlation_energy dictionary
    correlation_energy[method] = (results_h2[method] - E_HF) * units.mol/units.kJ

    print(method,"/",basis,": ", correlation_energy[method])

Correlation energy of H2 in kJ/mol:
RHF / cc-pVTZ :  0.0
MP2 / cc-pVTZ :  -83.13606113072339
CCD / cc-pVTZ :  11.35895804731748
CCSD / cc-pVTZ :  -103.26272488067134
CCSD(T) / cc-pVTZ :  -103.26272488067134
CISD / cc-pVTZ :  -103.26270363441932
PBE / cc-pVTZ :  -86.74495001703248
TPSS / cc-pVTZ :  -122.88322946841556
TPSSh / cc-pVTZ :  -121.56176930209637
B3LYP / cc-pVTZ :  -105.629567608503


### Exercise 2: $ \text{F}_2 $

First, built the molecule $ \text{F}_2 $.

In [13]:
# build the F2 molecule using Atoms
F2 = molecule("F2")

In [14]:
# view information about the molecules and visualize molecules (optional)
view(F2, viewer="x3d")

Calculate the energy of F2 using the methods CCSD, CCSD(T) and CCSD(T)-F12 and the basis sets cc-pVDZ, cc-VTZ, cc-VQZ and cc-V5Z. 

In [16]:
methods = ['CCSD','CCSD(T)']
bases = ['cc-pVDZ','cc-pVTZ','cc-pVQZ','cc-pV5Z']

results = {}
for method in methods:
    results[method] = {}

    for basis in bases:
        print("Running: ", method,"/",basis)

        # calculate potential energies. Store the results in a dictionary, where the keys are the method and the basis set
        F2.calc = ASEMolpro(ansatz=f"{method}/{basis}")
        results[method][basis] = F2.get_potential_energy()


method_f12 = 'CCSD(T)-F12b'
bases_f12 = ['cc-pVDZ-F12','cc-pVTZ-F12','cc-pVQZ-F12'] 

results_f12 = {}
results_f12[method_f12] = {}
for basis in bases_f12:
    print("Running: ", method_f12,"/",basis)

    # calculate potential energies. Store the results in a dictionary, where the keys are the method and the basis set
    F2.calc = ASEMolpro(ansatz=f"{method_f12}/{basis}")
    results_f12[method_f12][basis] = F2.get_potential_energy()


Running:  CCSD / cc-pVDZ
Running:  CCSD / cc-pVTZ
Running:  CCSD / cc-pVQZ
Running:  CCSD / cc-pV5Z
Running:  CCSD(T) / cc-pVDZ
Running:  CCSD(T) / cc-pVTZ
Running:  CCSD(T) / cc-pVQZ
Running:  CCSD(T) / cc-pV5Z
Running:  CCSD(T)-F12b / cc-pVDZ-F12
Running:  CCSD(T)-F12b / cc-pVTZ-F12
Running:  CCSD(T)-F12b / cc-pVQZ-F12


Calculate the correlation energy for each method and basis set. 

Note: The HF energy must be computed first for each basis set. After obtaining the HF energy, the correlation energy can be calculated for each method and basis set.


In [17]:
# HF calculation
hf_results = {}

for basis in bases:
    print("Running: HF /", basis)

    # calculate potential energies and store in a dictionary
    F2.calc = ASEMolpro(ansatz=f"HF/{basis}")
    hf_results[basis] = F2.get_potential_energy()

for basis in bases_f12:
    print("Running: HF /", basis)

    # calculate potential energies and store in a dictionary
    F2.calc = ASEMolpro(ansatz=f"HF/{basis}")
    hf_results[basis] = F2.get_potential_energy()

Running: HF / cc-pVDZ
Running: HF / cc-pVTZ
Running: HF / cc-pVQZ
Running: HF / cc-pV5Z
Running: HF / cc-pVDZ-F12
Running: HF / cc-pVTZ-F12
Running: HF / cc-pVQZ-F12


In [19]:
# correlation energy

corr_results = {}

for method in methods:
    corr_results[method] = {}

    for basis in bases:
        # save correlation energy for each method and basis set in dictionary
        corr_results[method][basis] = (results[method][basis] - hf_results[basis]) * units.mol/units.kJ

        print(method,"/",basis,": ", corr_results[method][basis])


CCSD / cc-pVDZ :  -1060.1977978817526
CCSD / cc-pVTZ :  -1383.7567550967153
CCSD / cc-pVQZ :  -1498.3672759746844
CCSD / cc-pV5Z :  -1541.4940233686548
CCSD(T) / cc-pVDZ :  -1084.378110256085
CCSD(T) / cc-pVTZ :  -1431.5745474006396
CCSD(T) / cc-pVQZ :  -1553.8117948761435
CCSD(T) / cc-pV5Z :  -1600.042705611682


In [21]:
# do the same for F12 methods
corr_results[method_f12] = {}

for basis in bases_f12:
    # save correlation energy for each method and basis set in dictionary
    corr_results[method_f12][basis] = (results_f12[method_f12][basis] - hf_results[basis]) * units.mol/units.kJ

    print(method_f12,"/",basis,": ", corr_results[method_f12][basis])


CCSD(T)-F12b / cc-pVDZ-F12 :  -1579.3230819441828
CCSD(T)-F12b / cc-pVTZ-F12 :  -1621.5613806709885
CCSD(T)-F12b / cc-pVQZ-F12 :  -1634.8196090317272


To estimate the correlation energy at the complete basis set (CBS) limit, a 2-point extrapolation can be used. For correlation energies, the standard formula is:

$ E_\mathrm{corr}^\mathrm{CBS} = \frac{n_2^3 E_\mathrm{corr}^\mathrm{2} - n_1^3 E_\mathrm{corr}^\mathrm{1}}{n_2^3 - n_1^3}  $

Indicies 1, 2 define method/basis set 1,2, e.g. 1 = CCSD(T)/cc-pVDZ, 2 = CCSD(T)/cc-pVQZ.

This method provides an approximate value of the correlation energy as if an infinitely large basis set were used, improving the accuracy of quantum chemical predictions.


In [23]:
def cbs_corr(E_corr1, E_corr2, n1, n2, alpha=3):
    """
    2 point CBS extrapolation for correlation energies
    E_corr1, E_corr2 : correlation energy of bases n1, n2
    n1, n2 : cardinal number of bases (DZ=2, TZ=3, QZ=4, 5Z=5)
    alpha : exponent (default 3)
    """
    return (n2**3 * E_corr2 - n1**3 * E_corr1) / (n2**3 - n1**3)

method = 'CCSD(T)'

n1 = 1
n2 = 2

basis1 = "cc-pVDZ"
basis2 = "cc-pVQZ"

# correlation energy = method - HF
E_corr_1 = results[method][basis1] - hf_results[basis1]
E_corr_2 = results[method][basis2] - hf_results[basis2]

E_corr_CBS = cbs_corr(E_corr_1, E_corr_2, n1, n2)

print("CBS correlation energy (CCSD(T)):", E_corr_CBS)

method_f12 = 'CCSD(T)-F12b'

n1 = 1
n2 = 2

basis1 = "cc-pVDZ-F12"
basis2 = "cc-pVTZ-F12"

E_corr_1 = results_f12[method_f12][basis1] - hf_results[basis1]
E_corr_2 = results_f12[method_f12][basis2] - hf_results[basis2]

E_corr_CBS_f12 = cbs_corr(E_corr_1, E_corr_2, n1, n2)

print("CBS correlation energy (CCSD(T)-F12b):", E_corr_CBS_f12)



CBS correlation energy (CCSD(T)): -16.799172489727425
CBS correlation energy (CCSD(T)-F12b): -16.868837726127303
